# 13.5 - Memory

**Phase:** 13 - LangGraph / Stateful Workflows

**Status:** VERIFIED

---

## 1. What Are We Solving?

Memory means persisting state across invocations and accumulating context within a workflow. LangGraph checkpointers save state after every node; `thread_id` isolates conversations.

## 2. Why Does This Matter?

## 3. Prerequisites

Units 13.1-13.4.

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Persist state across invocations with `MemorySaver`
- Accumulate history with the `add_messages` reducer
- Isolate conversations with thread IDs
- Bound memory so history does not grow forever

## 5. Mental Model

Memory is a notebook that persists:

```
within-run  -> lists/counters accumulated during one execution
cross-run   -> checkpoint state that survives between invocations
```

The graph reads and writes this notebook at every step.


## 6. Setup

In [1]:
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.checkpoint.memory import MemorySaver


## 7. Persistent Chatbot with Thread Isolation

Without a reducer the input value would simply replace the stored field. `add_messages` **appends** new messages to the checkpointed history.

In [2]:
class ChatState(TypedDict):
    msgs: Annotated[list, add_messages]


def chatbot(state):
    last = state["msgs"][-1]
    return {"msgs": [{"role": "assistant", "content": f"Echo: {last.content}"}]}


g = StateGraph(ChatState)
g.add_node("bot", chatbot)
g.add_edge(START, "bot")
g.add_edge("bot", END)
app = g.compile(checkpointer=MemorySaver())

alice = {"configurable": {"thread_id": "alice-1"}}
bob = {"configurable": {"thread_id": "bob-1"}}

app.invoke({"msgs": [{"role": "user", "content": "Hello"}]}, alice)
app.invoke({"msgs": [{"role": "user", "content": "Remind me: buy milk"}]}, alice)
app.invoke({"msgs": [{"role": "user", "content": "Hi from Bob"}]}, bob)

done = app.invoke({"msgs": [{"role": "user", "content": "What should I remember from us?"}]}, alice)
print("--- Alice thread ---")
for m in done["msgs"]:
    print(f"{m.type:>9}: {m.content}")
print("--- Bob thread ---")
print([m.content for m in app.get_state(bob).values["msgs"]])


--- Alice thread ---
    human: Hello
       ai: Echo: Hello
    human: Remind me: buy milk
       ai: Echo: Remind me: buy milk
    human: What should I remember from us?
       ai: Echo: What should I remember from us?
--- Bob thread ---
['Hi from Bob', 'Echo: Hi from Bob']


## 8. Accumulating Counter With a Custom Reducer

`Annotated[int, add]` merges each returned value into the persisted total instead of replacing it.

In [3]:
def add_int(a: int, b: int) -> int:
    return a + b


class CounterState(TypedDict):
    total: Annotated[int, add_int]
    last: int


def add_number(state):
    return {"total": state["last"]}


g = StateGraph(CounterState)
g.add_node("add", add_number)
g.add_edge(START, "add")
g.add_edge("add", END)
app = g.compile(checkpointer=MemorySaver())
cfg = {"configurable": {"thread_id": "counter-1"}}

for n in (5, 3, 10, 2):
    state = app.invoke({"total": 0, "last": n}, cfg)
    print(f"based on draft +{n} -> running total {state['total']}")


based on draft +5 -> running total 5
based on draft +3 -> running total 8
based on draft +10 -> running total 18
based on draft +2 -> running total 20


## 9. Bounded Memory: Keep the Last N Messages

A plain-list state with a trim step keeps memory from growing without limit.

In [4]:
class TrimState(TypedDict):
    msgs: list


def add_and_trim(state):
    user = state["msgs"][-1]
    state["msgs"] = (state["msgs"] + [{"role": "assistant", "content": f"Echo: {user}"}])[-4:]
    return state


g = StateGraph(TrimState)
g.add_node("add", add_and_trim)
g.add_edge(START, "add")
g.add_edge("add", END)
app = g.compile()

msgs = []
for i in range(6):
    msgs = app.invoke({"msgs": msgs + [{"role": "user", "content": f"msg {i}"}]})["msgs"]
    print(f"after msg {i}: kept {len(msgs)} messages")
print("contents:", [m["content"] for m in msgs])


after msg 0: kept 2 messages
after msg 1: kept 4 messages
after msg 2: kept 4 messages
after msg 3: kept 4 messages
after msg 4: kept 4 messages
after msg 5: kept 4 messages
contents: ['msg 4', "Echo: {'role': 'user', 'content': 'msg 4'}", 'msg 5', "Echo: {'role': 'user', 'content': 'msg 5'}"]



## Common Mistakes

- **Return `None` from a node** — the graph silently drops the update. Always `return state`.
- **Mutating state in the router** — routers must be side-effect free.
- **Forgetting the terminal condition** — cycles run forever without an iteration guard.
- **Typo in a state key** — `TypedDict` catches it at compile time; plain dicts do not.

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| `KeyError` on state | Field name mismatch | Match state keys to the `TypedDict` exactly |
| Graph won't compile | Node/referenced name typo | Check every string passed to `add_node`/`add_edge` |
| Node output ignored | Node returns `None` or a partial dict | Always `return state` (or a merge-able partial) |
| Infinite loop | No convergence guard | Add `max_steps` to state and check it in the router |
| Wrong branch taken | Router priority bug | Unit-test the router on every input variant |

## Best Practices

- Define all state fields upfront with defaults in a `TypedDict`.
- Keep node functions pure and focused: one responsibility each.
- Name nodes descriptively (`retrieve`, `generate`, not `step1`).
- Always add an iteration guard on loops.
- Inspect the graph with `app.get_graph().draw_mermaid()`.

## Hands-On Practice

1. **Basic:** Rerun the examples with new inputs; verify the trace.
2. **Guided:** Add a node that validates output before terminating.
3. **Independent:** Build a 3-step pipeline (fetch -> process -> summarize) with a retry node.
4. **Realistic:** Turn the example into a multi-department support agent.
5. **Challenge:** Save/load the state dict to JSON and resume the workflow from a checkpoint.

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.
